In [1]:
# Célula 1: Montar o Google Drive
print("Montando Google Drive...")
from google.colab import drive
drive.mount('/content/drive')
print("Google Drive montado com sucesso. Verifique a pasta 'drive/MyDrive'.")

Montando Google Drive...
Mounted at /content/drive
Google Drive montado com sucesso. Verifique a pasta 'drive/MyDrive'.


\## Instalar bibliotecas

### Subtask:
Instalar as bibliotecas necessárias para interagir com bancos de dados SQLite no Python.


In [13]:
import sqlite3
import pandas as pd

## Conectar ao banco de dados

### Subtask:
Estabelecer uma conexão com o arquivo do banco de dados localizado no Google Drive.


In [15]:
database_path = '/content/drive/MyDrive/00_Projeto_ICMC/database.db'
try:
    conn = sqlite3.connect(database_path)
    print(f"Conexão estabelecida com sucesso: {conn}")
except sqlite3.Error as e:
    print(f"Erro ao conectar ao banco de dados: {e}")
    conn = None # Ensure conn is None if connection fails

Conexão estabelecida com sucesso: <sqlite3.Connection object at 0x7bbe8661bb50>


## Mostra lista de tabelas no banco

In [16]:
# Define the SQL query
query = "SELECT name FROM sqlite_master WHERE type='table';"

# Create a cursor
cursor = conn.cursor()

# Execute the query
cursor.execute(query)

# Fetch all results
tables = cursor.fetchall()

# Print the table names
print("Tables in the database:")
for table in tables:
    print(table[0])

# Close the cursor
cursor.close()

Tables in the database:
Armazem
Cargo
Categoria_Produto
Centro_Custo
Cliente
Conta_Contabil
Departamento
Endereco_Cliente
Envio
Estoque
Fatura_Venda
Fornecedor
Funcionario
Funcionario_Cargo
Item_Pedido
Item_Receita
Lancamento_Contabil
Lancamento_Contabil_CC
Materia_Prima
Pagamento_Fornecedor
Pedido
Producao_Ordem
Produto
Projeto_TI
Recebimento_Materia_Prima
Receita_Produto
Rota_Entrega
Tarefa_TI


## Cria função que acessa o banco através de SQL

In [17]:
def run_query_and_display(query, connection):
  """
  Executa uma query SQL e exibe os resultados usando pandas.

  Args:
    query (str): A query SQL a ser executada.
    connection: O objeto de conexão com o banco de dados SQLite.
  """
  try:
    df = pd.read_sql_query(query, connection)
    display(df)
  except Exception as e:
    print(f"Erro ao executar a query: {e}")


## Exibir resultados

### Apresentação das queries para os casos de uso


In [19]:
# Caso de Uso 1: Demonstração e Execução dos Casos de Uso (Relatório Contábil)
query_example = """
SELECT
  LC.*,
  CC.*,
  CO.Nome_Conta
FROM Lancamento_Contabil AS LC
JOIN Lancamento_Contabil_CC AS LCC
  ON LC.ID_Lancamento = LCC.ID_Lancamento
JOIN Centro_Custo AS CC
  ON LCC.ID_Centro_Custo = CC.ID_Centro_Custo
JOIN Conta_Contabil AS CO
  ON LC.ID_Conta_Contabil = CO.ID_Conta_Contabil;
"""
print("Executing example query:")
run_query_and_display(query_example, conn)

Executing example query:


,ID_Lancamento,ID_Conta_Contabil,Data_Lancamento,Valor,Tipo_Lancamento,Referencia_Documento,ID_Centro_Custo,Nome_Centro_Custo,Nome_Conta
0,1,10001,2024-05-11,70000.0,Debito,901,1,Vendas,Ativos Circulantes - Caixa
1,2,10003,2024-05-11,70000.0,Credito,901,1,Vendas,Receita Bruta de Vendas
2,3,10001,2024-06-02,100000.0,Debito,903,1,Vendas,Ativos Circulantes - Caixa


In [20]:
# Caso de Uso 4: Demonstração e Execução dos Casos de Uso (Uso de Ontologia: Produtos Voadores)
query_example = """
SELECT T1.Nome_Produto, T1.Descricao_Produto
FROM Produto AS T1
INNER JOIN Categoria_Produto AS T2 ON T1.ID_Categoria_Produto = T2.ID_Categoria_Produto
WHERE T2.Nome_Categoria LIKE '%Voador%'
"""
print("Executing example query:")
run_query_and_display(query_example, conn)

Executing example query:


,Nome_Produto,Descricao_Produto
0,Mochila Voadora AeroPack,Mochila pessoal com propulsão a jato.


In [22]:
# Caso de Uso 6: Demonstração e Execução dos Casos de Uso (Conexões Diretas de Produto)
query_example = """
SELECT
  P.Nome_Produto,
  CP.Nome_Categoria,
  IP.ID_Pedido,
  IP.Quantidade,
  E.Quantidade_Disponivel,
  RP.ID_Receita,
  OP.ID_Ordem_Producao
FROM Produto AS P
JOIN Categoria_Produto AS CP
  ON P.ID_Categoria_Produto = CP.ID_Categoria_Produto
LEFT JOIN Item_Pedido AS IP
  ON P.ID_Produto = IP.ID_Produto
LEFT JOIN Estoque AS E
  ON P.ID_Produto = E.ID_Produto
LEFT JOIN Receita_Produto AS RP
  ON P.ID_Produto = RP.ID_Produto
LEFT JOIN Producao_Ordem AS OP
  ON P.ID_Produto = OP.ID_Produto;
"""
print("Executing example query:")
run_query_and_display(query_example, conn)

Executing example query:


,Nome_Produto,Nome_Categoria,ID_Pedido,Quantidade,Quantidade_Disponivel,ID_Receita,ID_Ordem_Producao
0,Quadriciclo X-Treme,Veículos,301.0,2.0,50.0,NaN,NaN
1,Quadriciclo X-Treme,Veículos,303.0,1.0,50.0,NaN,NaN
2,Mochila Voadora AeroPack,Equipamentos Voadores,302.0,1.0,10.0,NaN,NaN
3,Bike Montanha Carbono,Veículos,303.0,3.0,NaN,701.0,801.0
4,Capacete Adventure Pro,Acessórios,NaN,NaN,NaN,NaN,NaN


In [24]:
# Caso de Uso 9: Demonstração e Execução dos Casos de Uso  (Relatório Completo de Produto com Relações)
query_example = """
SELECT
  P.*,
  CP.*,
  R.*,
  IP.*,
  E.*,
  OP.*
FROM Produto AS P
INNER JOIN Categoria_Produto AS CP
  ON P.ID_Categoria_Produto = CP.ID_Categoria_Produto
LEFT JOIN Receita_Produto AS R
  ON P.ID_Produto = R.ID_Produto
LEFT JOIN Item_Pedido AS IP
  ON P.ID_Produto = IP.ID_Produto
LEFT JOIN Estoque AS E
  ON P.ID_Produto = E.ID_Produto
LEFT JOIN Producao_Ordem AS OP
  ON P.ID_Produto = OP.ID_Produto;
"""
print("Executing example query:")
run_query_and_display(query_example, conn)

Executing example query:


,ID_Produto,Nome_Produto,Descricao_Produto,Preco_Unitario_Venda,ID_Categoria_Produto,ID_Categoria_Produto,Nome_Categoria,ID_Receita,ID_Produto,ID_Item_Pedido,...,Preco_Venda,ID_Estoque,ID_Produto,Quantidade_Disponivel,Local_Armazenamento,ID_Ordem_Producao,ID_Produto,Quantidade_Produzir,Data_Inicio_Prevista,Status_Producao
0,201,Quadriciclo X-Treme,Veículo off-road de alta performance.,35000.0,1,1,Veículos,NaN,NaN,1.0,...,35000.0,1.0,201.0,50.0,Galpao A,NaN,NaN,NaN,None,None
1,201,Quadriciclo X-Treme,Veículo off-road de alta performance.,35000.0,1,1,Veículos,NaN,NaN,4.0,...,35000.0,1.0,201.0,50.0,Galpao A,NaN,NaN,NaN,None,None
2,202,Mochila Voadora AeroPack,Mochila pessoal com propulsão a jato.,120000.0,2,2,Equipamentos Voadores,NaN,NaN,2.0,...,120000.0,2.0,202.0,10.0,Galpao B,NaN,NaN,NaN,None,None
3,203,Bike Montanha Carbono,Bicicleta leve para trilhas agressivas.,8000.0,1,1,Veículos,701.0,203.0,3.0,...,8000.0,NaN,NaN,NaN,None,801.0,203.0,100.0,2024-07-01,Em Progresso
4,204,Capacete Adventure Pro,Capacete com sistema de comunicação.,1500.0,3,3,Acessórios,NaN,NaN,NaN,...,None,NaN,NaN,NaN,None,NaN,NaN,NaN,None,None
